# 🏛️ Kunstpixel Trondheim: Sensitiv Personprofilering & Mosaikk-innsamling (168 Ansatte)
### Hvordan tilsynelatende harmløse driftsdata i Trondheim avslører helse, privatøkonomi, bosted og habilitet

Denne Colab-notatblokken analyserer alle **168 simulerte ansatte** ved museet i **Trondheim** og demonstrerer hvordan en automatisert innsamlings- og koblingspipeline kan sette sammen åpne og interne kilder:

1. **Grunnlagsdata & Utvidet HR-profil (`hr_utvidet_profil_trondheim`):** 168 ansatte bosatt i Trondheim og omegn (Midtbyen, Bakklandet, Ila, Byåsen, Lade, Moholt, Tiller, Heimdal, Ranheim, Stjørdal og Melhus).
2. **Time-for-time vaktlogg (`vakt_time_for_time`):** 439 952 vakt-timer med eksakte klokkeslett og overtidsjusteringer.
3. **Kollektivavvik i Trøndelag (`kollektiv_avvik_trondheim`):** Stans på **SJ Nord Trønderbanen** (Hommelvik/Stjørdal), **Gråkallbanen** (Byåsen/Ila) og **AtB Metrobuss M1/M2** (Okstadbakken/Heimdal).
4. **Skattelister & Brønnøysund (`ekstern_osint_trondheim` + `hr_innleie`):** Kryssjekk av Virke-lønnstrinn mot faktisk nettoinntekt, gjeldsgrad og skjulte familiebånd til museets leverandører.


## 1. Koble til Google Drive eller lokal database
Kjører automatisk mot `colab_personvern_historikk.db` eller `ansatte_168_komplett_personvern_trondheim.csv`.


In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

# Hvis vi kjører i Google Colab, kan Google Drive monteres automatisk:
try:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
except ImportError:
    pass

mulige_stier = [
    "05_data",
    ".",
    "/content/drive/MyDrive/Kunstpixel/05_data",
    "/content"
]

data_dir = next((p for p in mulige_stier if os.path.exists(os.path.join(p, "ansatte_168_komplett_personvern_trondheim.csv")) or os.path.exists(os.path.join(p, "colab_personvern_historikk.db"))), ".")
db_path = os.path.join(data_dir, "colab_personvern_historikk.db")
csv_168_path = os.path.join(data_dir, "ansatte_168_komplett_personvern_trondheim.csv")

if os.path.exists(db_path):
    print(f"✅ Koblet til komplett SQLite-database: {db_path}")
    conn = sqlite3.connect(db_path)
    df_168 = pd.read_csv(csv_168_path) if os.path.exists(csv_168_path) else pd.read_sql_query("SELECT * FROM hr_utvidet_profil_trondheim", conn)
    df_vakt = pd.read_sql_query("SELECT * FROM vakt_time_for_time WHERE aar >= 2024", conn)
    df_innleie = pd.read_sql_query("SELECT * FROM hr_innleie", conn)
    df_kollektiv = pd.read_sql_query("SELECT * FROM kollektiv_avvik_trondheim", conn)
else:
    print(f"✅ Laster 168 ansatte fra CSV: {csv_168_path}")
    df_168 = pd.read_csv(csv_168_path)

print(f"Totalt antall ansatte analysert: {len(df_168)}")
df_168[["ansatt_id", "navn", "stilling", "bydel_kommune", "pendlerlinje_atb", "risikonivaa", "mosaikk_risikoscore"]].head(10)


✅ Koblet til komplett SQLite-database: 05_data\colab_personvern_historikk.db
Totalt antall ansatte analysert: 168


ansatt_id,navn,stilling,bydel_kommune,pendlerlinje_atb,risikonivaa,mosaikk_risikoscore
KP-ANS-001,Henrik Faye-Lund,Museumsdirektør,Midtbyen (Trondheim),Sykkel / Gange (Midtbyen),HØY,65
KP-ANS-002,Silje Lindqvist,Sjefkurator,Bakklandet (Trondheim),Sykkel / Gange (Bakklandet),MODERAT,50
KP-ANS-003,Gunnar Viken,Samlingsforvalter / Registrar,Stjørdal,SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),KRITISK,95
KP-ANS-004,Astrid Moen,Formidlingsleder,Tyholt / Moholt (Trondheim),AtB Metrobuss M3 (Moholt–Midtbyen),LAV,35
KP-ANS-005,Terje Bratt,Drifts- og eiendomssjef,Elgeseter / Øya (Trondheim),AtB Metrobuss M1/M2 (Sør–Midtbyen),KRITISK,95
KP-ANS-006,Jonas Berg,Seniorvert / Skrankeleder,Stjørdal,SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),LAV,35
KP-ANS-007,Ragnhild Eide,Kvalitets- og complianceansvarlig,Bakklandet (Trondheim),Sykkel / Gange (Bakklandet),MODERAT,50
KP-ANS-008,Guro Thorvaldsen,Museumspedagog,Stjørdal,SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),MODERAT,50
KP-ANS-009,Liv Eide,Sjefkurator,Midtbyen (Trondheim),Sykkel / Gange (Midtbyen),MODERAT,50
KP-ANS-010,Håkon Bratt,Assisterende direktør,Byåsen (Trondheim),Gråkallbanen (Lian–Byåsen–Ila–St. Olavs gt),KRITISK,85


## 2. Geografisk fordeling i Trondheim og Risikoprofil (Alle 168 ansatte)
Vi ser først på hvor de 168 ansatte bor i Trondheimsregionen og hvordan eksponeringsrisikoen fordeler seg når grunnlagsdataene kobles sammen.


In [2]:
print("=== FORDELING PER RISIKONIVÅ (168 ANSATTE) ===")
print(df_168["risikonivaa"].value_counts())

print("\n=== FORDELING PER BYDEL / KOMMUNE I TRONDHEIMSREGIONEN ===")
print(df_168["bydel_kommune"].value_counts())

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    df_168["bydel_kommune"].value_counts().plot(kind="barh", ax=axes[0], color="#1f77b4")
    axes[0].set_title("Bosted i Trondheim & omegn (168 ansatte)")
    axes[0].set_xlabel("Antall ansatte")

    df_168["risikonivaa"].value_counts().reindex(["KRITISK", "HØY", "MODERAT", "LAV"]).plot(
        kind="bar", ax=axes[1], color=["#d62728", "#ff7f0e", "#ffbb78", "#2ca02c"]
    )
    axes[1].set_title("Mosaikk-risikonivå ved krysskobling av kilder")
    axes[1].set_ylabel("Antall ansatte")
    plt.tight_layout()
    plt.show()
except ImportError:
    pass


=== FORDELING PER RISIKONIVÅ (168 ANSATTE) ===
risikonivaa
MODERAT    74
HØY        37
KRITISK    33
LAV        24
Name: count, dtype: int64

=== FORDELING PER BYDEL / KOMMUNE I TRONDHEIMSREGIONEN ===
bydel_kommune
Tiller (Trondheim)               24
Byåsen (Trondheim)               20
Stjørdal                         17
Ranheim (Trondheim)              16
Ila (Trondheim)                  14
Lade / Møllenberg (Trondheim)    14
Tyholt / Moholt (Trondheim)      12
Bakklandet (Trondheim)           11
Midtbyen (Trondheim)             10
Elgeseter / Øya (Trondheim)      10
Heimdal (Trondheim)              10
Melhus                           10
Name: count, dtype: int64


## 3. Kobling 1: Bosteds-avsløring via AtB Metrobuss, Gråkallbanen og Trønderbanen
Selv om en vakt- og fraværsliste er «anonymisert» (kun `ansatt_id`), kan vi kryssjekke datoer for egenmeldinger/forsinkelser mot kjente kollektivsammenbrudd i Trondheim:
- **SJ Nord Trønderbanen Nord:** Stengt ved Hommelvik/Leangen $\rightarrow$ peker ut pendlere fra **Stjørdal og Ranheim**.
- **AtB Metrobuss M1/M2:** Busskaos i Okstadbakken $\rightarrow$ peker ut ansatte på **Tiller og Heimdal**.
- **Gråkallbanen:** Strømstans ved Breidablikk $\rightarrow$ peker ut ansatte på **Byåsen og Ila**.


In [3]:
kollektiv_treff = df_168[df_168["kollektiv_avvik_kobling"] != "Ingen treff mot kollektivstans"]
print(f"🚨 Fant {len(kollektiv_treff)} ansatte hvis fravær/forsinkelse avslører eksakt pendlerkorridor i Trondheim:\n")
display(kollektiv_treff[[
    "ansatt_id", "navn", "bydel_kommune", "adresse_trondheim",
    "pendlerlinje_atb", "kollektiv_avvik_kobling"
]].head(12))


🚨 Fant 16 ansatte hvis fravær/forsinkelse avslører eksakt pendlerkorridor i Trondheim:


ansatt_id,navn,bydel_kommune,adresse_trondheim,pendlerlinje_atb,kollektiv_avvik_kobling
KP-ANS-023,Thea Larsen,Tiller (Trondheim),"Ivar Lykkes veg 73, 7075 Tiller (Trondheim)",AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (sykemelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-029,Henrik Helgesen,Byåsen (Trondheim),"Byåsveien 103, 7020 Byåsen (Trondheim)",Gråkallbanen (Lian–Byåsen–Ila–St. Olavs gt),TRD-AVV-03: Fravær (egenmelding) sammenfaller med 'Kjørestrømbrudd og sporvekselfeil ved Breidablikk / Kyvannet'
KP-ANS-031,Bjørn Viken,Stjørdal,"Kjøpmannsgata 28, 7500 Stjørdal",SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),TRD-AVV-01: Fravær (sykemelding) sammenfaller med 'Rasfare og stengt jernbanespor ved Hommelvik/Leangen under ekstremvær'
KP-ANS-041,Christian Pedersen,Melhus,"Melhusvegen 82, 7224 Melhus",SJ Nord Trønderbanen Sør (Melhus–Heimdal–Trondheim S),TRD-AVV-04: Fravær (sykemelding) sammenfaller med 'Signalfeil på Dovrebanen/Trønderbanen mellom Heimdal og Selsbakk'
KP-ANS-089,Per Johnsen,Byåsen (Trondheim),"Byåsveien 76, 7020 Byåsen (Trondheim)",Gråkallbanen (Lian–Byåsen–Ila–St. Olavs gt),TRD-AVV-03: Fravær (sykemelding) sammenfaller med 'Kjørestrømbrudd og sporvekselfeil ved Breidablikk / Kyvannet'
KP-ANS-102,Frida Lund,Tiller (Trondheim),"Ivar Lykkes veg 129, 7075 Tiller (Trondheim)",AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (egenmelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-103,Hilde Johnsen,Tiller (Trondheim),"Ivar Lykkes veg 26, 7075 Tiller (Trondheim)",AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (sykemelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-104,Geir Viken,Stjørdal,"Kjøpmannsgata 46, 7500 Stjørdal",SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),TRD-AVV-01: Fravær (sykemelding) sammenfaller med 'Rasfare og stengt jernbanespor ved Hommelvik/Leangen under ekstremvær'
KP-ANS-114,Liv Bratt,Tiller (Trondheim),"Ivar Lykkes veg 39, 7075 Tiller (Trondheim)",AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (egenmelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-125,Hege Nygård,Ranheim (Trondheim),"Ranheimsvegen 111, 7058 Ranheim (Trondheim)",SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),TRD-AVV-01: Fravær (sykemelding) sammenfaller med 'Rasfare og stengt jernbanespor ved Hommelvik/Leangen under ekstremvær'


## 4. Kobling 2: Avsløring av Helseopplysninger (GDPR Art. 9), Fagforening og Overtidsbrudd
Ved å kombinere stasjonsplassering time-for-time med fraværsmønster og overtidsjusteringer (`>= 10 timer`), kan algoritmen identifisere ansatte med rygg-/belastningslidelser, lysmigrene og høy utbrenthetsrisiko.


In [4]:
helse_utsatte = df_168[
    (df_168["helse_ergonomi_art9"] != "Ingen registrerte helsebegrensninger") &
    (df_168["sykefravaer_dager"] >= 20)
].sort_values(by="sykefravaer_dager", ascending=False)

print(f"⚠️ {len(helse_utsatte)} ansatte med eksponert Art. 9 helseprofil kombinert med høyt fravær og stasjonsslitasje:")
display(helse_utsatte[[
    "ansatt_id", "navn", "stilling", "primar_stasjon",
    "helse_ergonomi_art9", "fagforening_art9",
    "sykefravaer_dager", "overtid_aml_brudd"
]].head(12))


⚠️ 96 ansatte med eksponert Art. 9 helseprofil kombinert med høyt fravær og stasjonsslitasje:


ansatt_id,navn,stilling,primar_stasjon,helse_ergonomi_art9,fagforening_art9,sykefravaer_dager,overtid_aml_brudd
KP-ANS-001,Henrik Faye-Lund,Museumsdirektør,Administrasjonsfløyen 3. etg,Tinnitus / lydsensitivitet: Fritak fra barneverksteder og fullsatte skoleomvisninger,Econas,755,0
KP-ANS-010,Håkon Bratt,Assisterende direktør,Administrasjonsfløyen 3. etg,BHT-notat: Migrene utløst av flimrende spotbelysning i Sal C og Sal E,Uorganisert,731,0
KP-ANS-025,Maren Moen,HR- og organisasjonsrådgiver,Administrasjonsfløyen 3. etg,Astma/støvallergi: Skal unngå lengre arbeid i eldre magasin uten åndedrettsvern,Forskerforbundet,669,0
KP-ANS-031,Bjørn Viken,Assisterende direktør,Administrasjonsfløyen 3. etg,Tinnitus / lydsensitivitet: Fritak fra barneverksteder og fullsatte skoleomvisninger,Samfunnsviterne,656,0
KP-ANS-043,Sigrid Ruud,Kurator modernisme,Kuratorfløyen 2. etg,Tinnitus / lydsensitivitet: Fritak fra barneverksteder og fullsatte skoleomvisninger,Uorganisert,649,0
KP-ANS-032,Fredrik Nygård,Kurator samtidskunst,Kuratorfløyen 2. etg,Redusert nattesøvn / fritak fra seine kveldsvakter etter kl. 20:00 (BHT),Parat,625,0
KP-ANS-041,Christian Pedersen,Sjefkurator,Kuratorfløyen 2. etg,Senebetennelse/belastningsskade skulder etter tung utstillingsmontering,Forskerforbundet,591,0
KP-ANS-007,Ragnhild Eide,Kvalitets- og complianceansvarlig,Administrasjonsfløyen 3. etg,BHT-notat: Migrene utløst av flimrende spotbelysning i Sal C og Sal E,Uorganisert,530,0
KP-ANS-009,Liv Eide,Sjefkurator,Kuratorfløyen 2. etg,Senebetennelse/belastningsskade skulder etter tung utstillingsmontering,Parat,461,0
KP-ANS-002,Silje Lindqvist,Sjefkurator,Kuratorfløyen 2. etg,BHT-notat: Migrene utløst av flimrende spotbelysning i Sal C og Sal E,Forskerforbundet,457,0


## 5. Kobling 3: Skattelister vs. Virke-lønnstrinn & Brønnøysund-habilitet
Når museets interne lønnstrinn (Virke-tabellen) er kjent, kan hvem som helst kryssjekke dette mot offentlige skattelister og Brønnøysundregistrene:
1. **Positivt gap (Skatteliste $\gg$ Virke-lønn):** Avslører skjult bi-inntekt, konsulentvirksomhet eller utleie.
2. **Negativt gap (Skatteliste $\ll$ Virke-lønn):** Avslører store rentefradrag / tung privat gjeldsbelastning.
3. **Etternavn + Roller mot `hr_innleie`:** Avslører ansatte som bestiller eller godkjenner oppdrag fra egne familiemedlemmer!


In [5]:
habilitet_og_okonomi = df_168[
    (df_168["habilitet_flagg"] != "Ingen registrert konflikt") |
    (df_168["inntektsavvik_analyse"].str.contains("avslører"))
].sort_values(by="mosaikk_risikoscore", ascending=False)

print(f"🔍 Fant {len(habilitet_og_okonomi)} ansatte med betydelige økonomiske avvik eller habilitetsbånd:")
display(habilitet_og_okonomi[[
    "ansatt_id", "navn", "stilling", "virke_aarslonn_nok",
    "skatteliste_nettoinntekt", "skatteliste_nettoformue",
    "inntektsavvik_analyse", "habilitet_flagg"
]].head(15))


🔍 Fant 56 ansatte med betydelige økonomiske avvik eller habilitetsbånd:


ansatt_id,navn,stilling,virke_aarslonn_nok,skatteliste_nettoinntekt,skatteliste_nettoformue,inntektsavvik_analyse,habilitet_flagg
KP-ANS-031,Bjørn Viken,Assisterende direktør,972700,1366024,2803991,"Skatteliste er +510,048 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Nær familierelasjon til ekstern konsulent (45 kontrakter)
KP-ANS-146,Espen Ruud,Kurator modernisme,874600,1179753,4090956,"Skatteliste er +410,105 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Slektskap/styreverv hos fast kunsttransportør (49 kontrakter)
KP-ANS-104,Geir Viken,Forskningsstipendiat,831000,1214364,9162884,"Skatteliste er +483,084 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Nær familierelasjon til ekstern konsulent (45 kontrakter)
KP-ANS-114,Liv Bratt,Magasinforvalter,722000,1138588,3602639,"Skatteliste er +503,228 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Godkjenner egne driftskontrakter og har slekt i vakthold
KP-ANS-128,Kari Ruud,Assisterende direktør,983600,1403850,5208135,"Skatteliste er +538,282 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Slektskap/styreverv hos fast kunsttransportør (49 kontrakter)
KP-ANS-003,Gunnar Viken,Samlingsforvalter / Registrar,918200,1425849,8259701,"Skatteliste er +617,833 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Nær familierelasjon til ekstern konsulent (45 kontrakter)
KP-ANS-021,Per Ruud,Kommunikasjons- og markedssjef,907300,1240501,4948575,"Skatteliste er +442,077 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Slektskap/styreverv hos fast kunsttransportør (49 kontrakter)
KP-ANS-005,Terje Bratt,Drifts- og eiendomssjef,961800,1107697,6130852,"Skatteliste er +261,313 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Godkjenner egne driftskontrakter og har slekt i vakthold
KP-ANS-023,Thea Larsen,Museumspedagog,216600,834244,8725519,"Skatteliste er +643,636 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",Ingen registrert konflikt
KP-ANS-085,Andreas Ruud,Digital innholdsprodusent / Web,831000,1012056,9638499,"Skatteliste er +280,776 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring",OBS HABILITET: Slektskap/styreverv hos fast kunsttransportør (49 kontrakter)


## 6. Komplett Mosaikk-rapport: De mest eksponerte ansatte ved museet i Trondheim
Tabellen under viser hvordan enkeltbitene danner et komplett, høy-sensitivt personkort når alle kildene slås sammen.


In [6]:
kritiske = df_168[df_168["risikonivaa"] == "KRITISK"].sort_values(by="mosaikk_risikoscore", ascending=False)
print(f"🔴 Totalt {len(kritiske)} ansatte har KRITISK eksponeringsrisiko (Score >= 75/100):")
display(kritiske[[
    "ansatt_id", "navn", "fodselsdato", "adresse_trondheim",
    "stilling", "mosaikk_risikoscore", "avslort_sensitiv_innsikt"
]].head(20))


🔴 Totalt 33 ansatte har KRITISK eksponeringsrisiko (Score >= 75/100):


ansatt_id,navn,fodselsdato,adresse_trondheim,stilling,mosaikk_risikoscore,avslort_sensitiv_innsikt
KP-ANS-031,Bjørn Viken,1995-12-10,"Kjøpmannsgata 28, 7500 Stjørdal",Assisterende direktør,100,"k=1 på 'Administrasjonsfløyen 3. etg' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: Tinnitus / lydsensitivitet: Fritak fra barneverksteder og fullsatte skoleomvisninger (656 fraværsdager) | Geolokasjon avslørt via AtB/Trønderbanen: Bor i Stjørdal (7500) | OBS HABILITET: Nær familierelasjon til ekstern konsulent (45 kontrakter) | Skatteliste er +510,048 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring"
KP-ANS-128,Kari Ruud,1976-03-06,"Ilevollen 22, 7018 Ila (Trondheim)",Assisterende direktør,100,"k=1 på 'Administrasjonsfløyen 3. etg' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: Senebetennelse/belastningsskade skulder etter tung utstillingsmontering (185 fraværsdager) | Geolokasjon avslørt via AtB/Trønderbanen: Bor i Ila (Trondheim) (7018) | OBS HABILITET: Slektskap/styreverv hos fast kunsttransportør (49 kontrakter) | Skatteliste er +538,282 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring"
KP-ANS-146,Espen Ruud,1982-09-13,"Melhusvegen 34, 7224 Melhus",Kurator modernisme,100,"k=1 på 'Kuratorfløyen 2. etg' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: STAMI-tilrettelegging: Maks 2t sammenhengende ståtid (korsrygg/prolaps) (117 fraværsdager) | Geolokasjon avslørt via AtB/Trønderbanen: Bor i Melhus (7224) | OBS HABILITET: Slektskap/styreverv hos fast kunsttransportør (49 kontrakter) | Skatteliste er +410,105 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring"
KP-ANS-104,Geir Viken,1976-05-11,"Kjøpmannsgata 46, 7500 Stjørdal",Forskningsstipendiat,100,"k=1 på 'Kuratorfløyen 2. etg' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: Redusert nattesøvn / fritak fra seine kveldsvakter etter kl. 20:00 (BHT) (179 fraværsdager) | Geolokasjon avslørt via AtB/Trønderbanen: Bor i Stjørdal (7500) | OBS HABILITET: Nær familierelasjon til ekstern konsulent (45 kontrakter) | Skatteliste er +483,084 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring"
KP-ANS-114,Liv Bratt,1987-04-27,"Ivar Lykkes veg 39, 7075 Tiller (Trondheim)",Magasinforvalter,100,"k=1 på 'Magasin 1 (Klimastyrt lager)' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: BHT-notat: Migrene utløst av flimrende spotbelysning i Sal C og Sal E (71 fraværsdager) | Geolokasjon avslørt via AtB/Trønderbanen: Bor i Tiller (Trondheim) (7075) | OBS HABILITET: Godkjenner egne driftskontrakter og har slekt i vakthold | Skatteliste er +503,228 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring"
KP-ANS-003,Gunnar Viken,1974-04-23,"Kjøpmannsgata 109, 7500 Stjørdal",Samlingsforvalter / Registrar,95,"k=1 på 'Magasin 1 (Klimastyrt lager)' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: STAMI-tilrettelegging: Maks 2t sammenhengende ståtid (korsrygg/prolaps) (127 fraværsdager) | OBS HABILITET: Nær familierelasjon til ekstern konsulent (45 kontrakter) | Skatteliste er +617,833 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring | AML-risiko: 15 vakter med 10-11 timers varighet"
KP-ANS-005,Terje Bratt,1969-06-20,"Elgeseter gate 13, 7030 Elgeseter / Øya (Trondheim)",Drifts- og eiendomssjef,95,"k=1 på 'Vaktrom / Sikkerhetssentral' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: Senebetennelse/belastningsskade skulder etter tung utstillingsmontering (193 fraværsdager) | OBS HABILITET: Godkjenner egne driftskontrakter og har slekt i vakthold | Skatteliste er +261,313 kr høyere enn museets lønn -> avslører skjult bi-inntekt/næring | AML-risiko: 12 vakter med 10-11 timers varighet"
KP-ANS-023,Thea Larsen,1981-08-05,"Ivar Lykkes veg 73, 7075 Tiller (Trondheim)",Museumspedagog,95,"k=1 på 'Sal A (Mytologi og natur)' gir eksakt lokasjonssporing time-for-time | Art. 9 Helse: Senebetennelse/belastningsskade skulder etter tung utstillingsmontering (135 fraværsdager) | Geolokasj

## 7. Nærmere om ansatte «på kanten» (Hverdagsslitasje, tidsklemme og stram økonomi)
Her ser vi nøkternt på tre vanlige grupper som ligger i grenseland i arbeidshverdagen når vi kobler dataene:
1. **Slitasje og lange vakter:** Ansatte med mange lange vakter (10–11 timer), helse-/ergonominotat (f.eks. rygg eller migrene) og hyppig fravær.
2. **Tidsklemme og pendling:** Ansatte med småbarn under 12 år som pendler med Trønderbanen eller buss/trikk og får fravær ved kollektivstans.
3. **Stram økonomi eller ekstrajobb:** Ansatte i deltid eller med høy gjeldsbelastning i skattelisten, eller med bi-inntekt ved siden av museumsjobben.


In [7]:
print("--- A. SLITASJE & LANGE VAKTER (Overtid >= 12 vakter + Helse/Fravær) ---")
slitasje = df_168[
    (df_168["overtid_aml_brudd"] >= 12) &
    (df_168["sykefravaer_dager"] >= 25)
].sort_values(by=["overtid_aml_brudd", "sykefravaer_dager"], ascending=False)
display(slitasje[[
    "ansatt_id", "navn", "stilling", "stillingsprosent", "primar_stasjon",
    "overtid_aml_brudd", "sykefravaer_dager", "helse_ergonomi_art9"
]].head(10))

print("\n--- B. TIDSKLEMME (Småbarn + Pendling med kollektivavvik i Trondheim) ---")
tidsklemme = df_168[
    (df_168["antall_barn_under_12"] > 0) &
    (df_168["kollektiv_avvik_kobling"] != "Ingen treff mot kollektivstans")
].sort_values(by="antall_barn_under_12", ascending=False)
display(tidsklemme[[
    "ansatt_id", "navn", "bydel_kommune", "sivilstand",
    "antall_barn_under_12", "pendlerlinje_atb", "kollektiv_avvik_kobling"
]].head(10))

print("\n--- C. STRAM ØKONOMI / HØY GJELDSBELASTNING I SKATTELISTEN ---")
stram_okonomi = df_168[
    df_168["inntektsavvik_analyse"].str.contains("gjeldsgrad")
].sort_values(by="virke_aarslonn_nok", ascending=True)
display(stram_okonomi[[
    "ansatt_id", "navn", "stilling", "stillingsprosent",
    "virke_aarslonn_nok", "skatteliste_nettoinntekt", "sivilstand", "antall_barn_under_12"
]].head(10))


--- A. SLITASJE & LANGE VAKTER (Overtid >= 12 vakter + Helse/Fravær) ---

--- B. TIDSKLEMME (Småbarn + Pendling med kollektivavvik i Trondheim) ---

--- C. STRAM ØKONOMI / HØY GJELDSBELASTNING I SKATTELISTEN ---


ansatt_id,navn,stilling,stillingsprosent,primar_stasjon,overtid_aml_brudd,sykefravaer_dager,helse_ergonomi_art9
KP-ANS-023,Thea Larsen,Museumspedagog,0.3,Sal A (Mytologi og natur),17,135,Senebetennelse/belastningsskade skulder etter tung utstillingsmontering
KP-ANS-003,Gunnar Viken,Samlingsforvalter / Registrar,1.0,Magasin 1 (Klimastyrt lager),15,127,STAMI-tilrettelegging: Maks 2t sammenhengende ståtid (korsrygg/prolaps)
KP-ANS-005,Terje Bratt,Drifts- og eiendomssjef,1.0,Vaktrom / Sikkerhetssentral,12,193,Senebetennelse/belastningsskade skulder etter tung utstillingsmontering


ansatt_id,navn,bydel_kommune,sivilstand,antall_barn_under_12,pendlerlinje_atb,kollektiv_avvik_kobling
KP-ANS-146,Espen Ruud,Melhus,Gift,3,SJ Nord Trønderbanen Sør (Melhus–Heimdal–Trondheim S),TRD-AVV-04: Fravær (sykemelding) sammenfaller med 'Signalfeil på Dovrebanen/Trønderbanen mellom Heimdal og Selsbakk'
KP-ANS-029,Henrik Helgesen,Byåsen (Trondheim),Samboer,2,Gråkallbanen (Lian–Byåsen–Ila–St. Olavs gt),TRD-AVV-03: Fravær (egenmelding) sammenfaller med 'Kjørestrømbrudd og sporvekselfeil ved Breidablikk / Kyvannet'
KP-ANS-127,Liv Johansen,Ranheim (Trondheim),Enslig,2,SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),TRD-AVV-01: Fravær (sykemelding) sammenfaller med 'Rasfare og stengt jernbanespor ved Hommelvik/Leangen under ekstremvær'
KP-ANS-031,Bjørn Viken,Stjørdal,Samboer,2,SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),TRD-AVV-01: Fravær (sykemelding) sammenfaller med 'Rasfare og stengt jernbanespor ved Hommelvik/Leangen under ekstremvær'
KP-ANS-114,Liv Bratt,Tiller (Trondheim),Gift,2,AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (egenmelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-103,Hilde Johnsen,Tiller (Trondheim),Gift,2,AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (sykemelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-023,Thea Larsen,Tiller (Trondheim),Skilt,1,AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (sykemelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-102,Frida Lund,Tiller (Trondheim),Enslig,1,AtB Metrobuss M1/M2 (Tiller/Heimdal–Midtbyen),TRD-AVV-02: Fravær (egenmelding) sammenfaller med 'Snøkaos og sperret Okstadbakken / E6 Sør – leddbusser fast på tvers'
KP-ANS-126,Frida Tangen,Ranheim (Trondheim),Gift,1,SJ Nord Trønderbanen Nord (Stjørdal–Ranheim–Trondheim S),TRD-AVV-01: Fravær (sykemelding) sammenfaller med 'Rasfare og stengt jernbanespor ved Hommelvik/Leangen under ekstremvær'
KP-ANS-135,Espen Hansen,Byåsen (Trondheim),Enslig,1,Gråkallbanen (Lian–Byåsen–Ila–St. Olavs gt),TRD-AVV-03: Fravær (sykemelding) sammenfaller med 'Kjørestrømbrudd og sporvekselfeil ved Breidablikk / Kyvannet'


ansatt_id,navn,stilling,stillingsprosent,virke_aarslonn_nok,skatteliste_nettoinntekt,sivilstand,antall_barn_under_12
KP-ANS-082,Henrik Hagen,Kafémedarbeider (Deltid),0.4,227800,79730,Gift,0
KP-ANS-163,Liv Pettersen,Museumsvert og omviser (Deltid),0.4,262600,91910,Enslig,0
KP-ANS-154,Marit Sæther,Audioguide- og innholdsprodusent,0.4,288800,101080,Enslig,0
KP-ANS-073,Henrik Pettersen,Vekter og sikkerhetsvakt,0.6,407000,142450,Samboer,3
KP-ANS-037,Henrik Sandvik,Kaféleder,1.0,732900,256514,Enslig,0
KP-ANS-046,Jonas Nilsen,Sikkerhetsleder / RVR,1.0,765600,267960,Enslig,2
KP-ANS-145,Torstein Bakke,Senior museumspedagog,1.0,787400,275590,Samboer,0
KP-ANS-127,Liv Johansen,Administrasjonskonsulent,1.0,798300,279405,Enslig,2
KP-ANS-064,Erik Larsen,Forskningsstipendiat,1.0,809200,283220,Skilt,0
KP-ANS-019,Andreas Jensen,Kurator samtidskunst,1.0,841900,294665,Samboer,0
